# Imports

In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import time

import os
from tqdm import tqdm

from CustomLayers import DummyLinear
from benchmark.perplexity import measure_ppl

# Utils

In [70]:
def change_linear_layer(model, new_layer, device):
    for layer in model.model.layers:
        layer.self_attn.q_proj = new_layer(layer.self_attn.q_proj, device)
        layer.self_attn.k_proj = new_layer(layer.self_attn.k_proj, device)
        layer.self_attn.v_proj = new_layer(layer.self_attn.v_proj, device)
        layer.self_attn.o_proj = new_layer(layer.self_attn.o_proj, device)

        layer.mlp.gate_proj = new_layer(layer.mlp.gate_proj, device)
        layer.mlp.up_proj = new_layer(layer.mlp.up_proj, device)
        layer.mlp.down_proj = new_layer(layer.mlp.down_proj, device)

    model.lm_head = new_layer(model.lm_head, device)
    torch.cuda.empty_cache()

    return model

# Load Data

In [ ]:
# raw_datasets = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="test") # longer sequnces
raw_datasets = load_dataset('zhengxuanzenwu/wikitext-2-split-12', split='test')

Repo card metadata block was not found. Setting CardData to empty.


In [6]:
prompts = [x['text'] for x in raw_datasets if len(x['text']) > 0]
print('Number of sequences:', len(prompts))

Number of sequences: 8192


# Load Model

## Baseline

In [ ]:
model_id = 'unsloth/Llama-3.2-1B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map='cuda:0')

## Modified

In [8]:
custom_model = AutoModelForCausalLM.from_pretrained(model_id, device_map='cuda:1')
custom_model = change_linear_layer(custom_model, DummyLinear, custom_model.device)

# Perplexity benchmark

In [9]:
orig_ppl, orig_time = measure_ppl(prompts, model, tokenizer)

100%|██████████| 8192/8192 [07:43<00:00, 17.69it/s]


Perplexity: 345.0570
Mean time per sample: 0.057 s


In [10]:
custom_ppl, custom_time = measure_ppl(prompts, custom_model, tokenizer)

100%|██████████| 8192/8192 [08:02<00:00, 16.98it/s]



Perplexity: 345.0570
Mean time per sample: 0.059 s
